In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.naive_bayes import MultinomialNB, ComplementNB

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("../data/processed/v2/development_token_nohref_in_text.csv")

y = df["label"]
X = df.drop(columns=["label", "Id"])

text_col = "text"
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

# =========================
# COMMON PREPROCESSOR
# =========================
tfidf = TfidfVectorizer(
	max_features=50000,
	ngram_range=(1, 2),
	min_df=5
)

preprocess = ColumnTransformer(
	transformers=[
		("text", tfidf, text_col),
		("num", StandardScaler(), numeric_cols)
	]
)

# =========================
# MODELS TO TEST
# =========================
models = {
	"LogisticRegression": LogisticRegression(
		max_iter=1000,
		n_jobs=-1
	),
	"LinearSVC": LinearSVC(),
	"SGDClassifier": SGDClassifier(
		loss="log_loss",
		max_iter=1000,
		n_jobs=-1
	),
	"RandomForest": RandomForestClassifier(
		n_estimators=300,
		n_jobs=-1,
		random_state=42
	),
	"ExtraTrees": ExtraTreesClassifier(
		n_estimators=300,
		n_jobs=-1,
		random_state=42
	),
	"HistGradientBoosting": HistGradientBoostingClassifier(
		max_depth=6,
		random_state=42
	)
}

# Naive Bayes (TEXT ONLY)
nb_models = {
	"MultinomialNB": MultinomialNB(),
	"ComplementNB": ComplementNB()
}

cv = StratifiedKFold(
	n_splits=5,
	shuffle=True,
	random_state=42
)

results = []

# =========================
# FULL MODELS (text + num)
# =========================
for name, model in models.items():
	print(f"Running {name}...")
	pipeline = Pipeline([
		("prep", preprocess),
		("clf", model)
	])
	scores = cross_val_score(
		pipeline,
		X,
		y,
		cv=cv,
		scoring="f1_macro",
		n_jobs=-1
	)
	results.append((name, scores.mean()))

# =========================
# NAIVE BAYES (text only)
# =========================
for name, model in nb_models.items():
	print(f"Running {name}...")
	pipeline = Pipeline([
		("tfidf", tfidf),
		("clf", model)
	])
	scores = cross_val_score(
		pipeline,
		X[text_col],
		y,
		cv=cv,
		scoring="f1_macro",
		n_jobs=-1
	)
	results.append((name, scores.mean()))

# =========================
# RESULTS
# =========================
results_df = pd.DataFrame(
	results,
	columns=["model", "macro_f1"]
).sort_values("macro_f1", ascending=False)

print("\n=== MODEL COMPARISON ===")
print(results_df)


Running LogisticRegression...
Running LinearSVC...
Running SGDClassifier...
Running RandomForest...
Running ExtraTrees...
Running HistGradientBoosting...


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn\pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "C:\Users\msist\AppData\Roaming\Python\Python311\site-packages\sklearn